# 清理全城市购买市场条件 `city.csv`

这个 notebook 用来清理新导出的 `city.csv`。

这个文件的用途不是构造企业自身购买价格，而是构造 **全城市 × 产品** 的 purchase-side market condition。

也就是说，它用来替代之前样本内的 `city_buy.csv`。之前的城市层面数据只覆盖 3410 家样本企业；如果某个城市只有一家样本企业，那么旧的 market condition 会机械地等于这个企业自己的采购情况。现在这个 `city.csv` 应该代表城市中所有企业的购买情况，因此更适合作为外部市场条件。

清理逻辑与 `01_clean.ipynb` 一致：

1. 项目代码按字符串读入。
2. 只保留纯数字项目代码，删除乱码、`#N/A` 等异常代码。
3. 右补零到 19 位。
4. 取前 9 位作为 `product_id`。
5. 匹配 `bianma.dta` 中的合法 9 位产品码。
6. 删除金额或数量非正的记录。
7. 按 `city × product_id × year` 重新聚合，生成可用于回归的 market condition。


## Step 0 路径和基础设置

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE = Path(r'G:\Kuangyu_Temp\Outsource\productivity')
ROOT = BASE.parent

city_path = BASE / 'city.csv'
bianma_path = ROOT / 'bianma.dta'

# 明确命名为 full city market condition，避免和旧的样本内 city_buy.csv 混淆。
out_clean_csv = BASE / 'city_buy_full_clean.csv'
out_clean_dta = BASE / 'city_buy_full_clean.dta'
out_market_csv = BASE / 'market_conds_buy_full.csv'
out_market_dta = BASE / 'market_conds_buy_full.dta'

print('city_path =', city_path)
print('bianma_path =', bianma_path)
print('out_clean_csv =', out_clean_csv)
print('out_clean_dta =', out_clean_dta)
print('out_market_csv =', out_market_csv)
print('out_market_dta =', out_market_dta)

## Step 1 读入 `city.csv`

所有列先按字符串读入，避免项目代码被自动转成科学计数法或数值格式。

In [ ]:
na_values = ['', 'NULL', '(Null)', 'null', 'None', 'nan', 'NaN']

city_raw = pd.read_csv(city_path, dtype=str, na_values=na_values, encoding='utf-8-sig')
city_raw.columns = city_raw.columns.str.strip()

print('city_raw:', city_raw.shape)
print(list(city_raw.columns))
display(city_raw.head(10))

## Step 2 统一列名和变量类型

这里把 `购方地区` 统一成 `city`，因为后面要和企业购买面板按 `city product_id year` 合并。

In [ ]:
city = city_raw[['购方地区', '项目代码', '买方企业数', '金额合计', '数量合计']].copy()
city = city.rename(columns={
    '购方地区': 'city',
    '项目代码': 'product_code',
    '买方企业数': 'n_buyers_raw',
    '金额合计': 'value',
    '数量合计': 'qty'
})

city['city'] = city['city'].astype('string').str.strip()
city['product_code'] = city['product_code'].astype('string').str.strip()
city['n_buyers_raw'] = pd.to_numeric(city['n_buyers_raw'], errors='coerce')
city['value'] = pd.to_numeric(city['value'], errors='coerce')
city['qty'] = pd.to_numeric(city['qty'], errors='coerce')
city['year'] = 2017

print(city.shape)
display(city.head(10))
display(city[['n_buyers_raw', 'value', 'qty']].describe())

## Step 3 检查项目代码乱码情况

In [ ]:
n_total = len(city)
n_missing_code = city['product_code'].isna().sum()
n_numeric_code = city['product_code'].str.fullmatch(r'\d+', na=False).sum()
n_bad_code = n_total - n_missing_code - n_numeric_code

print('total rows:', n_total)
print('missing product_code:', n_missing_code)
print('pure numeric product_code:', n_numeric_code)
print('bad non-numeric product_code:', n_bad_code)
print('numeric share:', f'{n_numeric_code / n_total * 100:.2f}%')

bad_examples = city.loc[~city['product_code'].str.fullmatch(r'\d+', na=False), 'product_code'].dropna().drop_duplicates().head(30)
display(bad_examples.to_frame('bad_product_code_examples'))

## Step 4 清理项目代码、金额和数量

规则与主清理 notebook 保持一致：项目代码必须是纯数字；右补零到 19 位；取前 9 位作为 `product_id`。

In [ ]:
print('before cleaning:', len(city))

city_clean = city.dropna(subset=['city', 'product_code', 'value', 'qty']).copy()
print('after dropping missing city/product/value/qty:', len(city_clean))

city_clean = city_clean[city_clean['product_code'].str.fullmatch(r'\d+', na=False)].copy()
print('after keeping pure numeric product_code:', len(city_clean))

city_clean['code19'] = city_clean['product_code'].str.ljust(19, '0').str[:19]
city_clean['product_id'] = city_clean['code19'].str[:9]

city_clean = city_clean[(city_clean['value'] > 0) & (city_clean['qty'] > 0)].copy()
print('after keeping positive value and qty:', len(city_clean))
print('unique city:', city_clean['city'].nunique())
print('unique product_id before bianma match:', city_clean['product_id'].nunique())

display(city_clean.head(10))

## Step 5 匹配 `bianma.dta`

只保留合法 9 位产品码。

In [ ]:
bianma = pd.read_stata(bianma_path)
print('bianma:', bianma.shape, list(bianma.columns))

valid_products = set(bianma['product_id'].astype(str).str.strip())
print('valid products:', len(valid_products))

n_before_match = len(city_clean)
city_clean = city_clean[city_clean['product_id'].isin(valid_products)].copy()
print('matched rows:', len(city_clean), '/', n_before_match, f'({len(city_clean) / n_before_match * 100:.2f}%)')
print('unique product_id after bianma match:', city_clean['product_id'].nunique())

display(city_clean.head(10))

## Step 6 构造全城市 purchase-side market condition

输出数据的键是 `city product_id year`，可以直接和企业购买面板按 `product_id city year` 合并。

核心变量：

- `mkt_value`：该城市该产品的全市场购买金额。
- `mkt_qty`：该城市该产品的全市场购买数量。
- `p_mkt`：全城市市场平均采购价格，等于 `mkt_value / mkt_qty`。
- `ln_p_mkt`：市场平均采购价格对数。
- `ln_mkt_qty`：市场采购数量对数，衡量市场规模。
- `n_buyers`：买方企业数。
- `ln_n_buyers`：买方企业数对数，衡量市场厚度。

注意：如果 `city.csv` 是 SQL 在原始 `项目代码` 层面先 `COUNT(DISTINCT 购方企业ID)` 得到的，那么压缩到 9 位 `product_id` 后，这里把 `买方企业数` 加总可能会重复计算同一企业。严格做法是在 SQL 里直接按清理后的 9 位产品码统计 `COUNT(DISTINCT 购方企业ID)`。但金额、数量和价格变量可以在这里可靠加总。

In [ ]:
market_conds_buy = city_clean.groupby(['city', 'product_id', 'year'], as_index=False).agg(
    mkt_value=('value', 'sum'),
    mkt_qty=('qty', 'sum'),
    n_buyers=('n_buyers_raw', 'sum'),
    n_raw_codes=('product_code', 'nunique'),
    n_rows=('product_code', 'size')
)

market_conds_buy = market_conds_buy[(market_conds_buy['mkt_value'] > 0) & (market_conds_buy['mkt_qty'] > 0)].copy()
market_conds_buy['p_mkt'] = market_conds_buy['mkt_value'] / market_conds_buy['mkt_qty']
market_conds_buy['ln_p_mkt'] = np.log(market_conds_buy['p_mkt'])
market_conds_buy['ln_mkt_qty'] = np.log(market_conds_buy['mkt_qty'])
market_conds_buy['ln_n_buyers'] = np.log(market_conds_buy['n_buyers'])

market_conds_buy = market_conds_buy[['product_id', 'city', 'year', 'mkt_value', 'mkt_qty', 'p_mkt', 'ln_p_mkt', 'ln_mkt_qty', 'n_buyers', 'ln_n_buyers', 'n_raw_codes', 'n_rows']].copy()

print('market_conds_buy:', market_conds_buy.shape)
print('unique city:', market_conds_buy['city'].nunique())
print('unique product_id:', market_conds_buy['product_id'].nunique())
display(market_conds_buy.head(10))
display(market_conds_buy[['mkt_value', 'mkt_qty', 'n_buyers', 'p_mkt']].describe())

## Step 7 导出结果

这里不直接覆盖旧的 `market_conds.dta`，因为旧文件可能还含有 seller-side 的变量。先导出一个明确的 purchase-side full-market 文件，后续再决定是否与 seller full-market 文件合并后覆盖主回归用的 `market_conds.dta`。

In [ ]:
city_clean_export = city_clean[['city', 'product_code', 'code19', 'product_id', 'year', 'n_buyers_raw', 'value', 'qty']].copy()
market_conds_buy_export = market_conds_buy.copy()

city_clean_export.to_csv(out_clean_csv, index=False, encoding='utf-8-sig')
city_clean_export.to_stata(out_clean_dta, write_index=False, version=118)
market_conds_buy_export.to_csv(out_market_csv, index=False, encoding='utf-8-sig')
market_conds_buy_export.to_stata(out_market_dta, write_index=False, version=118)

print('saved:', out_clean_csv)
print('saved:', out_clean_dta)
print('saved:', out_market_csv)
print('saved:', out_market_dta)
print('city_clean_export:', city_clean_export.shape)
print('market_conds_buy_export:', market_conds_buy_export.shape)